# 01_EDA — Exploratory Data Analysis & Data Quality Audit
**Week 1 | Telco Customer Churn**

Covers:
1. Load data & fix TotalCharges blank-string bug
2. Collapse 'No internet service' pseudo-categories
3. Overall churn rate (class imbalance)
4. Churn by Contract type (H1)
5. Churn by Tenure bucket (H2)
6. Churn by Payment Method (H3)
7. Churn by OnlineSecurity (H4)
8. Churn by MonthlyCharges (H5)
9. Churn by SeniorCitizen (H6)
10. Churn by Services Count (H7)
11. CLV calculation
12. Hypothesis summary & save cleaned data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (8, 5)

df = pd.read_csv('../data/raw/telco_churn.csv')
print(f'Shape: {df.shape}')
df.head()

## Step 1 — Fix TotalCharges Blank-String Bug

TotalCharges loads as `object` because 11 rows contain `" "` (blank string) instead of NaN.
These are new customers with `tenure = 0`.

In [ ]:
# Show the bug
print('Before fix — dtype:', df['TotalCharges'].dtype)
blank_rows = df[df['TotalCharges'].str.strip() == '']
print(f'Blank string rows: {len(blank_rows)}')
print(blank_rows[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

In [ ]:
import numpy as np

df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print('After fix — dtype:', df['TotalCharges'].dtype)
print('Remaining NaNs:', df['TotalCharges'].isna().sum())

## Step 2 — Data Types Check

In [ ]:
df.info()

## Step 3 — Collapse 'No internet service' Categories

6 columns use 'No internet service' which is semantically identical to 'No'.
Reducing categories simplifies future modeling.

In [ ]:
service_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

for col in service_cols:
    df[col] = df[col].replace('No internet service', 'No')

print(df[service_cols].apply(lambda c: c.unique()))

## Step 4 — Overall Churn Rate

Expected: No ≈ 73.4%, Yes ≈ 26.6% → moderately imbalanced dataset.

In [ ]:
print(df['Churn'].value_counts(normalize=True) * 100)

sns.countplot(x='Churn', data=df)
plt.title('Overall Churn Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../reports/churn_distribution.png', dpi=150)
plt.show()

In [ ]:
# Binary target for groupby calculations
df['Churn_binary'] = (df['Churn'] == 'Yes').astype(int)

## H1 — Churn by Contract Type

Expected finding: Month-to-month customers churn the most. Two-year contracts churn the least.

Business Insight: Long-term contracts improve retention.

In [ ]:
print(pd.crosstab(df['Contract'], df['Churn'], normalize='index').round(3))

sns.countplot(x='Contract', hue='Churn', data=df)
plt.title('Churn by Contract Type')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../reports/churn_by_contract.png', dpi=150)
plt.show()

## H2 — Churn by Tenure Bucket

Expected finding: New customers (0-12 months) churn more frequently.

Business Insight: First-year customers require special attention.

In [ ]:
df['tenure_bucket'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-12', '13-24', '25-48', '49-72']
)

sns.countplot(x='tenure_bucket', hue='Churn', data=df)
plt.title('Churn by Tenure Bucket')
plt.ylabel('Count')
plt.xlabel('Tenure (months)')
plt.tight_layout()
plt.savefig('../reports/churn_by_tenure.png', dpi=150)
plt.show()

## H3 — Churn by Payment Method

Expected finding: Electronic Check customers show higher churn.

Business Insight: Payment experience may influence retention.

In [ ]:
sns.countplot(x='PaymentMethod', hue='Churn', data=df)
plt.title('Churn by Payment Method')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../reports/churn_by_payment.png', dpi=150)
plt.show()

## H4 — Churn by Online Security

Expected finding: Customers without OnlineSecurity churn significantly more.

In [ ]:
sns.countplot(x='OnlineSecurity', hue='Churn', data=df)
plt.title('Churn by Online Security')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../reports/churn_by_security.png', dpi=150)
plt.show()

## H5 — Churn by Monthly Charges

Expected finding: Churned customers pay higher monthly charges on average.

In [ ]:
print(df.groupby('Churn')['MonthlyCharges'].mean().round(2))

sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.title('Monthly Charges by Churn Status')
plt.tight_layout()
plt.savefig('../reports/churn_by_monthly_charges.png', dpi=150)
plt.show()

## H6 — Churn by Senior Citizen

Expected finding: Senior citizens have a higher churn rate.

In [ ]:
print(df.groupby('SeniorCitizen')['Churn_binary'].mean().round(3) * 100)

sns.countplot(x='SeniorCitizen', hue='Churn', data=df)
plt.title('Churn by Senior Citizen (0=No, 1=Yes)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../reports/churn_by_senior.png', dpi=150)
plt.show()

## H7 — Churn by Number of Services

Expected finding: Customers with more bundled services churn less (higher switching cost).

In [ ]:
df['services_count'] = df[service_cols].apply(lambda row: (row == 'Yes').sum(), axis=1)

print(df.groupby('services_count')['Churn_binary'].mean().round(3) * 100)

sns.countplot(x='services_count', hue='Churn', data=df)
plt.title('Churn by Number of Add-on Services')
plt.xlabel('Number of Services')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../reports/churn_by_services_count.png', dpi=150)
plt.show()

## CLV Calculation

CLV = MonthlyCharges × tenure

In [ ]:
df['CLV'] = df['MonthlyCharges'] * df['tenure']
print(df.groupby('Churn')['CLV'].describe().round(2))

## Hypothesis Summary

In [ ]:
hypotheses = {
    'H1: Month-to-month churn > annual churn':
        df[df['Contract']=='Month-to-month']['Churn_binary'].mean() >
        df[df['Contract']=='One year']['Churn_binary'].mean(),
    'H2: 0-12 mo tenure has highest churn':
        df.groupby('tenure_bucket', observed=True)['Churn_binary'].mean().idxmax() == '0-12',
    'H3: Electronic check has highest churn':
        df.groupby('PaymentMethod')['Churn_binary'].mean().idxmax() == 'Electronic check',
    'H4: No OnlineSecurity churns more':
        df[df['OnlineSecurity']=='No']['Churn_binary'].mean() >
        df[df['OnlineSecurity']=='Yes']['Churn_binary'].mean(),
    'H5: Higher MonthlyCharges → higher churn':
        df[df['Churn']=='Yes']['MonthlyCharges'].mean() >
        df[df['Churn']=='No']['MonthlyCharges'].mean(),
    'H6: Senior citizens churn more':
        df[df['SeniorCitizen']==1]['Churn_binary'].mean() >
        df[df['SeniorCitizen']==0]['Churn_binary'].mean(),
    'H7: More services → lower churn':
        df.groupby('services_count')['Churn_binary'].mean().is_monotonic_decreasing,
}

for h, result in hypotheses.items():
    status = '✅ CONFIRMED' if result else '❌ REJECTED'
    print(f'{status} — {h}')

## Save Cleaned Data

In [1]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/telco_churn_cleaned.csv', index=False)
print('✅ Cleaned data saved to data/processed/telco_churn_cleaned.csv')
print(f'   Rows: {len(df)} | Columns: {df.shape[1]}')

NameError: name 'df' is not defined